# HW2P2 on Kaggle: SE-ResNet-34 + ArcFace + weight EMA (run three)

Same code as the main notebook and `psc/train.py` (generated by `kaggle/make_kaggle_notebook.py`), adapted to Kaggle.

**Before running**, in the notebook editor's right-hand panel:
1. **Add-ons → Secrets**: add `KAGGLE_API_TOKEN` (your `KGAT_...` token, `KAGGLE_API_KEY` in `.env`) and tick it for this notebook.
   The competition is private, so it can't be added under Input; the first cell downloads it (about 5 minutes) with this token.
   *(Optional)* also add `WANDB_API_KEY` to log to wandb. Without it, wandb is disabled.
2. **Settings → Accelerator**: `GPU T4 x2` (both GPUs are used). `GPU P100` also works.
3. **Settings → Internet**: on. Needed for the data download and wandb.
4. *(Optional)* set `SUBMIT_TO_KAGGLE = True` below to submit `submission.csv` at the end with the same token.

**Run it** with **Save Version → Save & Run All (Commit)**. It runs in the background, and the
browser can be closed. Interactive sessions stop when idle, so they are only for the quick `SMOKE = True` check.

**12-hour limit.** The notebook measures the training speed first and lowers `epochs` so that training
plus inference fits in `time_budget_hours`. It stops early if an epoch runs slow, then still writes
`submission.csv` from the best checkpoint.

**Resuming in a second session** (only needed if a run stopped early): in a new version, add this
notebook's previous output as an input (**Add Input → Notebooks → Your Work**). The notebook copies
`checkpoints/<run_name>/` from it and continues from `last.pth` with the original epoch count.

**Outputs** (in `/kaggle/working`, under the version's **Output** tab): `submission.csv` and `checkpoints/<run_name>/`.

In [ ]:
# ---- Kaggle setup ----
import os
import sys
import glob
import time
import shutil

NOTEBOOK_START = time.time()  # the time budget counts from here

SMOKE = False             # True: 1 epoch on a slice of the data, no wandb, a few minutes end to end. Use it once before a full run.
SUBMIT_TO_KAGGLE = False  # True: submit submission.csv at the end (needs the KAGGLE_USERNAME / KAGGLE_KEY secrets)

ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:
    os.chdir("/kaggle/working")
sys.path.insert(0, os.getcwd())

def find_dirs(name, root="/kaggle/input", max_depth=5):
    """Folders called `name` under root. Depth-limited and skips image folders (a ** glob would list 480k images)."""
    hits = []
    for path, dirs, _ in os.walk(root):
        if name in dirs:
            hits.append(os.path.join(path, name))
        if path[len(root):].count(os.sep) >= max_depth:
            dirs.clear()
        dirs[:] = [d for d in dirs if d not in ("images", "ver_data", name)]
    return sorted(hits)

# Secrets (Add-ons -> Secrets)
def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

COMPETITION = "hw-2-p-2-fall-2026-student-competition"

def kaggle_api():
    """Kaggle API logged in with the KAGGLE_API_TOKEN secret (the KGAT_... token, KAGGLE_API_KEY in .env)."""
    import subprocess
    from importlib.metadata import version
    token = get_secret("KAGGLE_API_TOKEN")
    assert token, "Add your Kaggle API token as the secret KAGGLE_API_TOKEN (Add-ons -> Secrets) and attach it to this notebook"
    os.environ["KAGGLE_API_TOKEN"] = token
    if tuple(int(x) for x in version("kaggle").split(".")[:2]) < (1, 8):  # KGAT_ tokens need kaggle >= 1.8
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle==1.8.0"], check=True)
    import kaggle  # authenticates on import
    return kaggle.api

# Locate the competition data: the folder that holds cls_data/ and ver_data/.
# The competition is private, so it can't be attached as an input: download it with the API instead.
DOWNLOAD_DIR = "/tmp/hw2p2"  # large scratch disk; /kaggle/working would put 5+ GB of images into the saved output
_hits = find_dirs("cls_data") + find_dirs("cls_data", DOWNLOAD_DIR) + sorted(glob.glob("./hw2p2_data/cls_data"))
if not _hits:
    import subprocess
    print("Downloading the competition data (a few minutes)...")
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    kaggle_api().competition_download_files(COMPETITION, path=DOWNLOAD_DIR, quiet=False)
    _zip = os.path.join(DOWNLOAD_DIR, COMPETITION + ".zip")
    subprocess.run(["unzip", "-qo", _zip, "-d", DOWNLOAD_DIR], check=True)
    os.remove(_zip)
    _hits = find_dirs("cls_data", DOWNLOAD_DIR)
assert _hits, f"Competition data not found in {DOWNLOAD_DIR}"
DATA_ROOT = os.path.dirname(_hits[0])
print("Data root:", DATA_ROOT, os.listdir(DATA_ROOT))

WANDB_KEY = None if SMOKE else get_secret("WANDB_API_KEY")
if WANDB_KEY is None:
    os.environ["WANDB_MODE"] = "disabled"
    print("wandb disabled (smoke test, or no WANDB_API_KEY secret)")

os.system("nvidia-smi")

## Imports

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
try:
    from torchsummary import summary
except ImportError:  # not installed on Kaggle: the summary is optional
    summary = lambda model, size: print(model)
import torchvision
from torchvision.io import decode_image
from torchvision.utils import make_grid
from torchvision import transforms
import torchvision.transforms.v2 as T
import os
import gc
from tqdm import tqdm
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn import metrics as mt
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import glob
import wandb
import matplotlib.pyplot as plt
import csv
import pandas as pd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", DEVICE, "| GPUs:", torch.cuda.device_count())
torch.backends.cudnn.benchmark = True  # fixed 112x112 inputs: let cuDNN pick the fastest kernels

## Config
The defaults are run three. The comment under `config` lists the switches that reproduce run two.

In [ ]:
import json

config = {
    'run_name': 'run3_seresnet34_ema', # Each run gets its own checkpoint folder: checkpoints/<run_name>/
    'data_root': DATA_ROOT,
    'batch_size': 256, # Increase this if your GPU can handle it
    'lr': 0.1,
    'warmup_epochs': 2, # Linear LR warmup, then cosine decay to 0 (stepped every batch)
    'epochs': 70,
    'num_classes': 8631, # Dataset contains 8631 classes for classification, reduce this number if you want to train on a subset, but only for train dataset and not on val dataset
    'augument': True,
    'random_erasing': 0.25, # Probability of erasing a patch (simulates occlusion)
    'grayscale': 0.1, # Probability of converting to grayscale (lighting/colour robustness)
    'weight_decay': 5e-4, # Not applied to BatchNorm / bias parameters
    'label_smoothing': 0.0,
    # Model: ResNet-34 layout adapted to 112x112 faces (7x7 final map), BN embedding head
    'blocks': [3, 4, 6, 3],
    'channels': [64, 128, 256, 512],
    'embedding_dim': 512,
    'dropout': 0.2, # Before the embedding projection
    'se': True, # Squeeze-and-Excitation channel attention in every residual block (writeup 4.4, SEResNet)
    'drop_path': 0.1, # Stochastic depth: max prob. of skipping a residual branch (deepest block), linear from 0
    'ema_decay': 0.9998, # Exponential moving average of the weights (~5k-step horizon); EMA weights are evaluated and submitted
    # Loss: ArcFace additive angular margin, margin ramped up from 0 over the first epochs
    'arcface_s': 32.0,
    'arcface_m': 0.4,
    'margin_warmup_epochs': 5,
    'time_budget_hours': 11.0, # Kaggle stops sessions at 12h; epochs are lowered to fit training + inference in this budget
    'resume': True, # Continue from checkpoints/<run_name>/last.pth if it exists (PSC sessions are capped at 8h)
    'num_workers': 0 if os.name == 'nt' else min(8, len(os.sched_getaffinity(0))), # Windows notebooks can't pickle notebook-defined Datasets
}
# To reproduce run two instead: run_name='run2_resnet34_arcface', epochs=50, grayscale=0, se=False, drop_path=0, ema_decay=0
config['checkpoint_dir'] = os.path.join("./checkpoints", config['run_name'])
os.makedirs(config['checkpoint_dir'], exist_ok=True)
with open(os.path.join(config['checkpoint_dir'], "run_config.json"), "w") as f:
    json.dump(config, f, indent=2)

In [ ]:
# ---- Smoke test and resume from a previous Kaggle version ----
if SMOKE:  # separate folder, 1 epoch, never resumes the real run
    config.update(run_name='smoke_test', epochs=1, warmup_epochs=1, margin_warmup_epochs=1, resume=False)
    config['checkpoint_dir'] = os.path.join("./checkpoints", config['run_name'])
    os.makedirs(config['checkpoint_dir'], exist_ok=True)

# A previous version's output attached as input: copy its checkpoint folder for this run (never overwrites)
RESUMING = False
if config['resume']:
    _prev = [p for d in find_dirs("checkpoints") for p in glob.glob(os.path.join(d, config['run_name'], "last.pth"))]
    if _prev:
        for _f in glob.glob(os.path.join(os.path.dirname(_prev[0]), "*")):
            _dst = os.path.join(config['checkpoint_dir'], os.path.basename(_f))
            if os.path.isfile(_f) and not os.path.exists(_dst):
                shutil.copy(_f, _dst)
        print("Copied previous checkpoints from", os.path.dirname(_prev[0]))
    _last = os.path.join(config['checkpoint_dir'], "last.pth")
    if os.path.exists(_last):
        RESUMING = True
        import torch
        _saved = torch.load(_last, map_location="cpu", weights_only=False)["config"]
        config['epochs'] = _saved['epochs']  # the LR schedule was built for this length
        print(f"Resuming {config['run_name']}: original run length {config['epochs']} epochs")
        del _saved

## Data

In [ ]:
def create_transforms(image_size: int = 112, augment: bool = True) -> T.Compose:
    """Create transform pipeline for face recognition."""

    # Step 1: Basic transformations
    transform_list = [
        # Resize the image to the desired size (image_size x image_size)
        T.Resize((image_size, image_size)),

        # Convert PIL Image to tensor
        T.ToTensor(),

        # Convert image to float32 and scale the pixel values to [0, 1]
        T.ToDtype(torch.float32, scale=True),
    ]

    # Step 2: Data augmentation (optional, based on `augment` argument)
    if augment:  # This block will be executed if `augment=True`
        # Faces stay faces under small crops/scales, mirroring and lighting changes.
        # No vertical flip: upside-down faces do not occur in the test data.
        transform_list.extend([
            T.RandomResizedCrop(image_size, scale=(0.8, 1.0), ratio=(0.9, 1.1), antialias=True),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        ])
        if config.get('grayscale', 0) > 0:
            transform_list.append(T.RandomGrayscale(p=config['grayscale']))

    # Step 3: Standard normalization for image recognition tasks
    # The Normalize transformation requires mean and std values for each channel (R, G, B).
    # Here, we are normalizing the pixel values to have a mean of 0.5 and std of 0.5 for each channel.
    transform_list.extend([
        T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Standard mean and std for face recognition tasks
    ])

    # Random erasing works on the normalized tensor, so it goes last.
    # It blanks a small patch, mimicking occlusion (glasses, hands, hair).
    if augment and config['random_erasing'] > 0:
        transform_list.append(T.RandomErasing(p=config['random_erasing'], scale=(0.02, 0.15)))

    # Return the composed transformation pipeline
    return T.Compose(transform_list)

In [ ]:
train_transforms = create_transforms(augment=config['augument'])
val_transforms   = create_transforms(augment=False)

In [ ]:
class ImageDataset(Dataset):
    """Custom dataset for loading images with optional labels."""

    def __init__(self, root, transform, num_classes=None, preload=False):
        """
        Args:
            root (str): Dataset root directory.
                Expected structure:
                  - labeled:
                        root/
                          images/
                          labels.txt   (img label)
                  - unlabeled:
                        root/
                          images/
            transform (callable): Transform applied to images.
            num_classes (int, optional): Number of classes to keep (labeled only).
            preload (bool): Preload images into memory.
        """
        self.root = root
        self.transform = transform
        self.preload = preload

        self.image_paths = []
        self.labels = None      # None → unlabeled dataset
        self.classes = None
        self.images = []

        labels_file = os.path.join(self.root, "labels.txt")
        images_dir = os.path.join(self.root, "images")

        # --------------------------------------------------
        # Detect labeled vs unlabeled dataset
        # --------------------------------------------------
        has_labels = os.path.exists(labels_file)

        if has_labels:
            self.labels = []
            self.classes = set()

            with open(labels_file, "r") as f:
                lines = f.readlines()

            # Sort by label (keeps your original behavior)
            lines = sorted(lines, key=lambda x: int(x.strip().split(" ")[-1]))

            all_labels = sorted(set(int(line.strip().split(" ")[1]) for line in lines))

            if num_classes is not None:
                selected_classes = set(all_labels[:num_classes])
            else:
                selected_classes = set(all_labels)

            for line in tqdm(lines, desc="Loading labeled dataset"):
                img_path, label = line.strip().split(" ")
                label = int(label)

                if label in selected_classes:
                    self.image_paths.append(os.path.join(images_dir, img_path))
                    self.labels.append(label)
                    self.classes.add(label)

            self.classes = sorted(self.classes)

            assert len(self.image_paths) == len(self.labels), "Images and labels mismatch!"

        else:
            # --------------------------------------------------
            # Unlabeled dataset: load all images from images/
            # --------------------------------------------------
            image_files = sorted(os.listdir(images_dir))

            for img in tqdm(image_files, desc="Loading unlabeled dataset"):
                self.image_paths.append(os.path.join(images_dir, img))

        # --------------------------------------------------
        # Preload images if requested
        # --------------------------------------------------
        if self.preload:
            self.images = [
                decode_image(p, mode="RGB") / 255.0
                for p in tqdm(self.image_paths, desc="Preloading images")
            ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        if self.preload:
            image = self.images[idx]
        else:
            image = decode_image(self.image_paths[idx], mode="RGB") / 255.0

        image = self.transform(image)

        # Return with or without label
        if self.labels is not None:
            return image, self.labels[idx]
        else:
            return image

In [ ]:
gc.collect()
cls_data_dir = config['data_root'] + '/cls_data'

# Datasets
# TODO: Implement the datasets for training, validation, and testing
cls_train_dataset = ImageDataset(cls_data_dir + '/train', train_transforms, num_classes=config['num_classes'])
cls_val_dataset   = ImageDataset(cls_data_dir + '/dev', val_transforms, num_classes=config['num_classes'])
cls_test_dataset  = ImageDataset(cls_data_dir + '/test', val_transforms)

assert cls_train_dataset.classes == cls_val_dataset.classes, "Class mismatch!"

# Dataloaders
# TODO: Implement the dataloaders for training, validation, and testing
loader_kwargs = dict(batch_size=config['batch_size'], num_workers=config['num_workers'], pin_memory=True, persistent_workers=config['num_workers'] > 0)
cls_train_loader = DataLoader(cls_train_dataset, shuffle=True, drop_last=True, **loader_kwargs) # drop_last: BatchNorm1d can't train on a final batch of 1
cls_val_loader   = DataLoader(cls_val_dataset, shuffle=False, **loader_kwargs)
cls_test_loader  = DataLoader(cls_test_dataset, shuffle=False, **loader_kwargs)

In [ ]:
if SMOKE:  # 1/200 of train, 1/20 of dev
    from torch.utils.data import Subset
    cls_train_loader = DataLoader(Subset(cls_train_dataset, range(0, len(cls_train_dataset), 200)), shuffle=True, drop_last=True, **loader_kwargs)
    cls_val_loader = DataLoader(Subset(cls_val_dataset, range(0, len(cls_val_dataset), 20)), shuffle=False, **loader_kwargs)

In [ ]:
class ImagePairDataset(Dataset):
    """Custom dataset for loading and transforming image pairs (labeled or unlabeled)."""

    def __init__(self, root, pairs_file, transform, preload=False):
        """
        Args:
            root (str): Path to the directory containing the images.
            pairs_file (str): File with:
                - img1 img2 label   (labeled)
                - img1 img2         (unlabeled)
            transform (callable): Transform to apply to images.
            preload (bool): Whether to preload images into memory.
        """
        self.root = root
        self.transform = transform
        self.preload = preload

        self.image1_paths = []
        self.image2_paths = []
        self.matches = None  # None → unlabeled dataset

        # Read file
        with open(pairs_file, "r") as f:
            lines = f.readlines()

        # Detect whether labels exist
        first_cols = lines[0].strip().split()
        has_labels = len(first_cols) == 3

        if has_labels:
            self.matches = []

        for line in tqdm(lines, desc="Loading dataset"):
            parts = line.strip().split()

            if has_labels:
                img1, img2, match = parts
                self.matches.append(int(match))
            else:
                img1, img2 = parts

            self.image1_paths.append(os.path.join(self.root, img1))
            self.image2_paths.append(os.path.join(self.root, img2))

        assert len(self.image1_paths) == len(self.image2_paths)
        if has_labels:
            assert len(self.matches) == len(self.image1_paths)

        # Preload images if requested
        if self.preload:
            self.image1_cache = [
                decode_image(p, mode="RGB") / 255.0
                for p in tqdm(self.image1_paths, desc="Preloading image1")
            ]
            self.image2_cache = [
                decode_image(p, mode="RGB") / 255.0
                for p in tqdm(self.image2_paths, desc="Preloading image2")
            ]

    def __len__(self):
        return len(self.image1_paths)

    def __getitem__(self, idx):
        # Load images
        if self.preload:
            img1 = self.image1_cache[idx]
            img2 = self.image2_cache[idx]
        else:
            img1 = decode_image(self.image1_paths[idx], mode="RGB") / 255.0
            img2 = decode_image(self.image2_paths[idx], mode="RGB") / 255.0

        img1 = self.transform(img1)
        img2 = self.transform(img2)

        # Return with or without label
        if self.matches is not None:
            return img1, img2, self.matches[idx]
        else:
            return img1, img2

In [ ]:
gc.collect()
ver_data_dir = config['data_root'] + '/ver_data'

# Datasets
# TODO: Implement the datasets for validation and testing
find_pairs = lambda tag: sorted(glob.glob(os.path.join(config['data_root'], '**', f'*{tag}*pair*.txt'), recursive=True))[0]
ver_val_dataset  = ImagePairDataset(ver_data_dir, find_pairs('val'), val_transforms)
ver_test_dataset = ImagePairDataset(ver_data_dir, find_pairs('test'), val_transforms)

# Dataloader
# TODO: Implement the dataloaders for validation and testing
ver_val_loader   = DataLoader(ver_val_dataset, shuffle=False, **loader_kwargs)
ver_test_loader  = DataLoader(ver_test_dataset, shuffle=False, **loader_kwargs)

## Model

In [ ]:
class SEModule(nn.Module):
    """
    Squeeze-and-Excitation: global-average-pool each channel ("squeeze"), pass through a small
    bottleneck MLP and a sigmoid ("excitation"), then rescale the channels. Lets the block
    emphasize the feature maps that matter for this face. ~1% extra parameters.
    """

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x)


def drop_path(x, p: float, training: bool):
    """Stochastic depth: during training, zero the whole residual branch for a random subset of samples."""
    if not training or p == 0.0:
        return x
    keep = 1.0 - p
    mask = x.new_empty(x.shape[0], 1, 1, 1).bernoulli_(keep)
    return x * mask / keep


class BasicBlock(nn.Module):
    """
    ResNet basic block: conv3x3-BN-ReLU-conv3x3-BN plus identity skip.
    When the block downsamples (stride 2) or widens, the skip uses a 1x1 strided conv + BN.
    """

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, se: bool = False, drop_path: float = 0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.se = SEModule(out_ch) if se else None
        self.drop_path = drop_path

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.se is not None:
            out = self.se(out)
        out = drop_path(out, self.drop_path, self.training)
        return self.relu(out + self.shortcut(x))


class NormLinear(nn.Module):
    """
    Cosine classifier: logits = s * cos(theta) between the L2-normalized embedding and
    L2-normalized class weights. The ArcFace loss adds the angular margin during training.
    """

    def __init__(self, in_features: int, num_classes: int, s: float):
        super().__init__()
        self.s = s
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        # fp32 outside autocast: ArcFace needs precise cosines near 1
        with torch.autocast(device_type=x.device.type, enabled=False):
            cos = F.linear(F.normalize(x.float(), dim=1), F.normalize(self.weight.float(), dim=1))
        return self.s * cos


class Network(nn.Module):
    """
    ResNet-34-style backbone for 112x112 faces.

    112 --stem s1--> 112 --stage1 s2--> 56 --stage2 s2--> 28 --stage3 s2--> 14 --stage4 s2--> 7
    The stem keeps full resolution (faces are small; run one's stride-4 stem threw detail away),
    and each stage halves resolution while doubling channels (constant compute per stage).
    """

    def __init__(self, num_classes: int, blocks=(3, 4, 6, 3), channels=(64, 128, 256, 512),
                 embedding_dim: int = 512, dropout: float = 0.2, s: float = 32.0,
                 se: bool = False, drop_path: float = 0.0):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(inplace=True),
        )

        stages = []
        in_ch = channels[0]
        # Stochastic depth grows linearly with depth: early blocks are almost never dropped
        n_total = sum(blocks)
        dp = [drop_path * i / max(1, n_total - 1) for i in range(n_total)]
        k = 0
        for n_blocks, out_ch in zip(blocks, channels):
            layers = []
            for b in range(n_blocks):
                layers.append(BasicBlock(in_ch if b == 0 else out_ch, out_ch, stride=2 if b == 0 else 1, se=se, drop_path=dp[k]))
                k += 1
            stages.append(nn.Sequential(*layers))
            in_ch = out_ch
        self.stages = nn.Sequential(*stages)

        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten())

        # Embedding head: Dropout -> Linear -> BN1d (the BN-normalized vector is the face embedding)
        self.embedding = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_ch, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
        )
        self.cls_layer = NormLinear(embedding_dim, num_classes, s=s)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        # Zero-init the last BN of each residual branch so every block starts as identity
        for m in self.modules():
            if isinstance(m, BasicBlock):
                nn.init.zeros_(m.bn2.weight)

    def forward(self, x, return_feats: bool = False):
        """
        Args:
            x (Tensor): Input image batch of shape (B, 3, H, W)
            return_feats (bool): If True, return only feature embeddings

        Returns:
            dict with keys:
                - 'feats': feature embeddings (used for verification)
                - 'out': classification logits, s * cos(theta)
        """
        feats = self.embedding(self.pool(self.stages(self.stem(x))))

        if return_feats:
            return feats

        out = self.cls_layer(feats)
        return {"feats": feats, "out": out}


class FlipTTA(nn.Module):
    """
    Test-time augmentation for inference: runs the model on the image and its mirror.
    Embeddings are summed (after L2-normalizing each) and logits averaged.
    Exposes the same {'feats', 'out'} dict, so the course submission utils work unchanged.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    @torch.no_grad()
    def forward(self, x):
        a = self.model(x)
        b = self.model(torch.flip(x, dims=[3]))
        feats = F.normalize(a["feats"], dim=1) + F.normalize(b["feats"], dim=1)
        return {"feats": feats, "out": (a["out"] + b["out"]) / 2}


# Initialize model
model = Network(
    num_classes=config["num_classes"],
    blocks=config["blocks"],
    channels=config["channels"],
    embedding_dim=config["embedding_dim"],
    dropout=config["dropout"],
    s=config["arcface_s"],
    se=config.get("se", False),
    drop_path=config.get("drop_path", 0.0),
).to(DEVICE).to(memory_format=torch.channels_last)

# Kaggle's T4 x2: split each batch across both GPUs. Checkpoints still store the plain model.
train_model = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model

n_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_params:,}")
assert n_params < 30_000_000, "Over the 30M parameter limit"

# Optional: model summary (eval mode: summary libraries feed a dummy batch as small as 1,
# and the BatchNorm1d in the embedding head can't train on a single sample)
model.eval()
summary(model, (3, 112, 112))
model.train()

In [ ]:
# ---- Fit the run into Kaggle's time limit ----
# Time a few training steps (forward + backward, data loading included), then lower `epochs`
# so that training + validation + inference fit in time_budget_hours. The weights are restored afterwards.
import copy

INFERENCE_RESERVE_S = 20 * 60  # loading the best checkpoint, test inference, writing the CSVs

if not SMOKE and not RESUMING:
    _state = copy.deepcopy(model.state_dict())  # BN running stats change in train mode
    train_model.train()
    _it = iter(cls_train_loader)
    _n_warm, _n_timed = 10, 40
    for _i in range(_n_warm + _n_timed):
        if _i == _n_warm:
            torch.cuda.synchronize()
            _t0 = time.time()
        _x, _y = next(_it)
        _x, _y = _x.to(DEVICE, non_blocking=True), _y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type='cuda'):
            _loss = F.cross_entropy(train_model(_x)["out"].float(), _y)
        _loss.backward()
        model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    _s_per_step = (time.time() - _t0) / _n_timed
    del _it, _x, _y, _loss
    model.load_state_dict(_state)
    del _state

    _train_s = _s_per_step * len(cls_train_loader)
    # Validation sees ~10% as many images as training, at roughly 3x the speed (no backward, no augmentation)
    _val_s = _s_per_step * (len(cls_val_loader) + 2 * len(ver_val_loader)) / 3
    _epoch_s = 1.05 * (_train_s + _val_s)
    _left_s = config['time_budget_hours'] * 3600 - (time.time() - NOTEBOOK_START) - INFERENCE_RESERVE_S
    _fit = max(1, int(_left_s // _epoch_s))
    print(f"{_s_per_step:.3f} s/step -> about {_epoch_s / 60:.1f} min/epoch; {_left_s / 3600:.2f}h left fits {_fit} epochs")
    if _fit < config['epochs']:
        print(f"Lowering epochs from {config['epochs']} to {_fit} (warmups unchanged)")
        config['epochs'] = _fit
        with open(os.path.join(config['checkpoint_dir'], "run_config.json"), "w") as f:
            json.dump(config, f, indent=2)
    gc.collect()
    torch.cuda.empty_cache()

## Loss, optimizer, EMA, scheduler

In [ ]:
import math

# ---------------------------------------------------
# Loss Function
# ---------------------------------------------------

class ArcFaceLoss(nn.Module):
    """
    Additive angular margin loss (ArcFace, writeup section 5.6.2).
    Takes the cosine-classifier logits s*cos(theta) and replaces the target logit
    with s*cos(theta_y + m), so the true class must win by an angular margin m.
    `m` is a plain attribute so the training loop can ramp it up.
    """

    def __init__(self, s: float, m: float, label_smoothing: float = 0.0):
        super().__init__()
        self.s = s
        self.m = m
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, logits, labels):
        cos = (logits.float() / self.s).clamp(-1 + 1e-7, 1 - 1e-7)
        cos_m, sin_m = math.cos(self.m), math.sin(self.m)
        sin = torch.sqrt(1.0 - cos * cos)
        phi = cos * cos_m - sin * sin_m  # cos(theta + m)
        # When theta + m > pi, cos(theta + m) stops being monotonic; fall back to a CosFace-style penalty
        phi = torch.where(cos > math.cos(math.pi - self.m), phi, cos - math.sin(math.pi - self.m) * self.m)

        one_hot = F.one_hot(labels, num_classes=cos.size(1)).bool()
        margin_logits = self.s * torch.where(one_hot, phi, cos)
        return self.ce(margin_logits, labels)


# ArcFace already enforces a margin on clean one-hot targets; label smoothing / mixup / cutmix conflict with it
assert config['label_smoothing'] == 0.0, "Do not use label smoothing with ArcFace"
criterion = ArcFaceLoss(s=config['arcface_s'], m=0.0, label_smoothing=config['label_smoothing'])

def margin_at(epoch):
    """Linearly ramp the ArcFace margin from 0 to arcface_m over margin_warmup_epochs."""
    warm = config['margin_warmup_epochs']
    return config['arcface_m'] * min(1.0, (epoch + 1) / warm) if warm > 0 else config['arcface_m']

# ---------------------------------------------------
# Optimizer
# ---------------------------------------------------

# SGD with Nesterov momentum. Weight decay only on conv/linear weights:
# decaying BatchNorm scales and biases fights normalization and hurts accuracy.
decay, no_decay = [], []
for name, p in model.named_parameters():
    (no_decay if p.ndim <= 1 else decay).append(p)
optimizer = optim.SGD(
    [{'params': decay, 'weight_decay': config['weight_decay']},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=config['lr'], momentum=0.9, nesterov=True,
)

# ---------------------------------------------------
# Weight EMA
# ---------------------------------------------------

# Keeps an exponential moving average of the weights (and BatchNorm statistics), updated every step.
# The average sits in a flatter, better-generalizing region than the last SGD iterate; it is what we
# validate, checkpoint as the best model and submit. The raw model keeps training as usual.
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(config['ema_decay']), use_buffers=True) if config.get('ema_decay') else None

# ---------------------------------------------------
# Learning Rate Scheduler
# ---------------------------------------------------

# Linear warmup then cosine decay to 0, stepped once per BATCH (smoother than per epoch).
# Warmup keeps the high LR from destabilizing the freshly initialized deep network.
steps_per_epoch = len(cls_train_loader)
warmup_steps = config['warmup_epochs'] * steps_per_epoch
total_steps = config['epochs'] * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ---------------------------------------------------
# Mixed-Precision Training (FP16)
# ---------------------------------------------------

# GradScaler enables stable mixed-precision training on supported GPUs (e.g., T4, V100).
# This reduces memory usage and can significantly speed up training.
scaler = torch.amp.GradScaler(device='cuda')

## Metrics and train / validation loops

In [ ]:
class AverageMeter:
    """
    Tracks and computes the running average of a scalar metric.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val, n: int = 1):
        self.val = float(val)
        self.sum += val * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, topk=(1,)):
    """
    Computes top-k accuracy for the given logits and targets.

    Args:
        logits (Tensor): Model outputs of shape (B, C)
        targets (Tensor): Ground-truth labels of shape (B,)
        topk (tuple): Values of k for top-k accuracy

    Returns:
        List[Tensor]: Accuracy values in percentage for each k
    """
    maxk = min(max(topk), logits.size(1))
    batch_size = targets.size(0)

    _, preds = logits.topk(maxk, dim=1, largest=True, sorted=True)
    preds = preds.t()
    correct = preds.eq(targets.view(1, -1))

    accuracies = []
    for k in topk:
        k = min(k, maxk)
        correct_k = correct[:k].reshape(-1).float().sum(0)
        accuracies.append(correct_k * 100.0 / batch_size)

    return accuracies


def verification_metrics(labels, scores, fpr_targets=None):
    """
    Computes standard verification metrics: ACC, EER, AUC, and TPR@FPR.

    Args:
        labels (list or array): Binary labels (0 or 1)
        scores (list or array): Similarity scores
        fpr_targets (list, optional): Target FPR values for reporting TPR

    Returns:
        dict: Verification metrics
    """
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    # TODO: Compute verificaion metrics
    # Hint: Read the documentation for sklearn.metrics.roc_curve
    # and use it to compute fpr and tpr
    fpr, tpr, _ = mt.roc_curve(labels, scores, pos_label=1)

    # Hint: You can use interp1d from scipy.interpolate to create roc_interp
    roc_interp = interp1d(fpr, tpr, bounds_error=False, fill_value=(0.0, 1.0))

    # Equal Error Rate (EER)
    # Hint: You might find brentq from scipy.optimize helpful to compute EER
    eer = 100.0 * brentq(lambda x: 1.0 - x - roc_interp(x), 0.0, 1.0)

    # Area Under Curve (AUC)
    # Hint: Use mt.auc to compute AUC
    # Be careful to multiply by 100.0 to express as percentage
    auc = 100.0 * mt.auc(fpr, tpr)

    # Accuracy (best threshold)
    # TODO: Compute TNR, pos_count, neg_count, and accuracy
    # Hint: Use fpr and tpr to compute TNR
    # Hint: pos_count is the number of positive labels (1s)
    # Hint: neg_count is the number of negative labels (0s)
    tnr = 1.0 - fpr
    pos_count = np.sum(labels == 1)
    neg_count = np.sum(labels == 0)
    acc = 100.0 * float(np.max(tpr * pos_count + tnr * neg_count)) / len(labels)

    # TPR @ specific FPRs
    tpr_at_fpr = []
    if fpr_targets is not None:
        for fpr_val in fpr_targets:
            tpr_at_fpr.append(
                (f"TPR@FPR={fpr_val}", 100.0 * float(roc_interp(fpr_val)))
            )

    return {
        "ACC": acc,
        "EER": eer,
        "AUC": auc,
        "TPRs": tpr_at_fpr,
    }

In [ ]:
def train_epoch(
    model,
    dataloader,
    optimizer,
    scaler,
    device,
    criterion,
    scheduler=None,
    ema=None,
):
    """
    Runs one training epoch.

    Args:
        model (nn.Module): Model to train
        dataloader (DataLoader): Training dataloader
        optimizer (Optimizer): Optimizer
        scaler (GradScaler): AMP gradient scaler
        device (torch.device): Training device
        criterion (callable): Loss function
        scheduler (LRScheduler, optional): Stepped once per batch
        ema (AveragedModel, optional): Weight EMA, updated after every optimizer step

    Returns:
        tuple: (avg_accuracy, avg_loss)
    """
    model.train()

    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    progress = tqdm(
        dataloader,
        desc="Train",
        dynamic_ncols=True,
        leave=False,
    )

    for images, labels in progress:
        optimizer.zero_grad(set_to_none=True)

        # Move data to device
        images = images.to(device, non_blocking=True)

        if isinstance(labels, (tuple, list)):
            # For mixup / cutmix style labels
            targets1, targets2, lam = labels
            labels = (
                targets1.to(device, non_blocking=True),
                targets2.to(device, non_blocking=True),
                lam,
            )
        else:
            labels = labels.to(device, non_blocking=True)

        # Forward pass (mixed precision)
        with torch.amp.autocast(device_type = 'cuda'):
            outputs = model(images)
            logits = outputs["out"]
            loss = criterion(logits, labels)

        # Backward + optimizer step (AMP-safe)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        if ema is not None:
            ema.update_parameters(model)

        # Metrics
        batch_loss = loss.item()
        loss_meter.update(batch_loss)

        with torch.no_grad():
            batch_acc = topk_accuracy(logits, labels, topk=(1,))[0].item()
            acc_meter.update(batch_acc)

        # Progress bar update
        progress.set_postfix(
            loss=f"{batch_loss:.4f} ({loss_meter.avg:.4f})",
            acc=f"{batch_acc:.2f}% ({acc_meter.avg:.2f}%)",
            lr=f"{optimizer.param_groups[0]['lr']:.6f}",
        )

    return acc_meter.avg, loss_meter.avg

In [ ]:
@torch.no_grad()
def valid_epoch_cls(
    model,
    dataloader,
    device,
    criterion,
):
    """
    Runs one validation epoch for classification.

    Args:
        model (nn.Module): Trained model
        dataloader (DataLoader): Validation dataloader
        device (torch.device): Evaluation device
        criterion (callable): Loss function

    Returns:
        tuple: (avg_accuracy, avg_loss)
    """
    model.eval()

    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    progress = tqdm(
        dataloader,
        desc="Val (Cls)",
        dynamic_ncols=True,
        leave=False,
    )

    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward pass (inference-only)
        outputs = model(images)
        logits = outputs["out"]
        loss = criterion(logits, labels)

        # Metrics
        batch_loss = loss.item()
        loss_meter.update(batch_loss)

        batch_acc = topk_accuracy(logits, labels, topk=(1,))[0].item()
        acc_meter.update(batch_acc)

        # Progress bar update
        progress.set_postfix(
            loss=f"{batch_loss:.4f} ({loss_meter.avg:.4f})",
            acc=f"{batch_acc:.2f}% ({acc_meter.avg:.2f}%)",
        )

    return acc_meter.avg, loss_meter.avg

In [ ]:
@torch.no_grad()
def valid_epoch_ver(model, pair_dataloader, device, fpr_targets=None):
    """
    Runs one validation epoch for verification (image pairs).

    Args:
        model (nn.Module): Trained model
        pair_dataloader (DataLoader): DataLoader yielding (img1, img2, label) tuples
        device (torch.device): Evaluation device
        fpr_targets (list, optional): List of FPRs for TPR reporting

    Returns:
        dict: Verification metrics including ACC, EER, AUC, TPR@FPR
    """
    model.eval()
    scores = []
    match_labels = []

    progress = tqdm(
        pair_dataloader,
        desc="Val (Veri)",
        dynamic_ncols=True,
        leave=False,
    )

    for images1, images2, labels in progress:
        # Move data to device
        images1 = images1.to(device, non_blocking=True)
        images2 = images2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward pass
        images = torch.cat([images1, images2], dim=0)
        outputs = model(images)
        feats = F.normalize(outputs["feats"], dim=1)

        # Split features and compute similarity
        feats1, feats2 = feats.chunk(2, dim=0)
        similarity = F.cosine_similarity(feats1, feats2, dim=1)

        # Accumulate results
        scores.append(similarity.cpu().numpy())
        match_labels.append(labels.cpu().numpy())

        progress.update()

    # Concatenate batch-wise results
    scores = np.concatenate(scores)
    match_labels = np.concatenate(match_labels)

    # Default FPR targets if not provided
    if fpr_targets is None:
        fpr_targets = [1e-4, 5e-4, 1e-3, 5e-3, 5e-2]

    # Compute verification metrics
    metric_dict = verification_metrics(match_labels, scores, fpr_targets)

    # Optional: print metrics
    print("Verification Metrics:", metric_dict)

    return metric_dict

## wandb

In [ ]:
if WANDB_KEY:
    wandb.login(key=WANDB_KEY) # API Key is in your wandb account, under settings (wandb.ai/settings)

In [ ]:
# Create your wandb run. The run id is stored in the run's checkpoint folder,
# so re-running this cell in a new PSC session resumes the same wandb run.
wandb_id_path = os.path.join(config['checkpoint_dir'], "wandb_run_id.txt")
if config['resume'] and os.path.exists(wandb_id_path):
    wandb_run_id = open(wandb_id_path).read().strip()
else:
    wandb_run_id = wandb.util.generate_id()
    with open(wandb_id_path, "w") as f:
        f.write(wandb_run_id)

run = wandb.init(
    name = config['run_name'],
    id = wandb_run_id,
    resume = "allow",
    project = "hw2p2", ### Project should be created in your wandb account
    config = config ### Wandb Config for your run
)

## Checkpointing

In [ ]:
# Ensure checkpoint directory exists (one folder per run: checkpoints/<run_name>/)
checkpoint_dir = config["checkpoint_dir"]
os.makedirs(checkpoint_dir, exist_ok=True)


def save_model(model, optimizer, scheduler, metrics, epoch, path, ema=None):
    """
    Saves model, optimizer, scheduler, and training metrics to a checkpoint.

    Args:
        model (nn.Module): Model to save
        optimizer (Optimizer): Optimizer
        scheduler (LRScheduler): Learning rate scheduler
        metrics (dict): Dictionary of tracked metrics
        epoch (int): Current epoch
        path (str): Path to save checkpoint
        ema (AveragedModel, optional): Weight EMA; its weights are stored as 'ema_state_dict'
    """
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "metrics": metrics,
            "epoch": epoch,
            "config": config,
            "ema_state_dict": ema.module.state_dict() if ema is not None else None,
        },
        path,
    )
    print(f"Checkpoint saved at {path}")


def load_model(model, optimizer=None, scheduler=None, path="./checkpoint.pth", device=None):
    """
    Loads model, optimizer, scheduler, and metrics from a checkpoint.

    Args:
        model (nn.Module): Model to load weights into
        optimizer (Optimizer, optional): Optimizer to load state
        scheduler (LRScheduler, optional): Scheduler to load state
        path (str): Path to checkpoint file
        device (torch.device, optional): Device mapping for checkpoint

    Returns:
        tuple: model, optimizer, scheduler, epoch, metrics
    """
    map_location = device if device is not None else "cpu"
    checkpoint = torch.load(path, map_location=map_location)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    epoch = checkpoint.get("epoch", 0)
    metrics = checkpoint.get("metrics", {})

    print(f"Checkpoint loaded from {path} (epoch {epoch})")
    return model, optimizer, scheduler, epoch, metrics

## Training

In [ ]:
# Training setup
start_epoch = 0
best_cls_acc = 0.0
best_ret_acc = 0.0
best_combined_score = 0.0

# Resume from this run's last checkpoint (never from another run's folder)
last_path = os.path.join(config["checkpoint_dir"], "last.pth")
if config["resume"] and os.path.exists(last_path):
    model, optimizer, scheduler, last_epoch, last_metrics = load_model(
        model, optimizer, scheduler, path=last_path, device=DEVICE
    )
    if ema is not None:
        ckpt = torch.load(last_path, map_location="cpu")  # CPU: don't hold a second copy of the checkpoint on the GPU
        ema.module.load_state_dict(ckpt["ema_state_dict"])
        del ckpt
        ema.n_averaged.fill_(1)  # >0, so the next update averages instead of overwriting with the raw weights
    start_epoch = last_epoch + 1
    best_cls_acc = last_metrics.get("best_cls_acc", 0.0)
    best_ret_acc = last_metrics.get("best_ret_acc", 0.0)
    best_combined_score = last_metrics.get("best_combined_score", 0.0)
    print(f"Resuming {config['run_name']} at epoch {start_epoch + 1}")

epoch_time = None
for epoch in range(start_epoch, config["epochs"]):
    # Stop before Kaggle's 12h limit: leave room for this epoch (with slack) and for inference
    elapsed = time.time() - NOTEBOOK_START
    if epoch_time is not None and elapsed + 1.15 * epoch_time + INFERENCE_RESERVE_S > config["time_budget_hours"] * 3600:
        print(f"Time budget reached after {elapsed / 3600:.2f}h: stopping before epoch {epoch + 1}. "
              "Resume in a new version with this output attached as input (see the top of the notebook).")
        break
    epoch_t0 = time.time()
    print(f"\n=== Epoch {epoch + 1}/{config['epochs']} ===")

    curr_lr = optimizer.param_groups[0]["lr"]
    criterion.m = margin_at(epoch)
    # -----------------------------
    # Train (scheduler steps every batch inside train_epoch)
    # -----------------------------
    train_cls_acc, train_loss = train_epoch(
        train_model, cls_train_loader, optimizer, scaler, DEVICE, criterion, scheduler, ema
    )
    print(
        f"Train | Cls Acc: {train_cls_acc:.4f}% | Loss: {train_loss:.4f} | LR: {curr_lr:.6f} | Margin: {criterion.m:.3f}"
    )

    metrics = {
        "train_cls_acc": train_cls_acc,
        "train_loss": train_loss,
        "lr": curr_lr,
        "arcface_m": criterion.m,
    }

    # Validate (and pick the best checkpoint by) the EMA weights when EMA is on
    eval_model = ema.module if ema is not None else model

    # -----------------------------
    # Classification Validation
    # -----------------------------
    valid_cls_acc, valid_loss = valid_epoch_cls(
        eval_model, cls_val_loader, DEVICE, criterion
    )

    print(f"Val (Cls) | Acc: {valid_cls_acc:.4f}% | Loss: {valid_loss:.4f}")

    metrics.update({
        "valid_cls_acc": valid_cls_acc,
        "valid_loss": valid_loss,
    })

    # -----------------------------
    # Verification / Retrieval Validation
    # -----------------------------
    valid_ret_metrics = valid_epoch_ver(eval_model, ver_val_loader, DEVICE)
    valid_ret_acc = valid_ret_metrics["ACC"]
    valid_eer = valid_ret_metrics["EER"]
    print(f"Val (Veri) | ACC: {valid_ret_acc:.4f}% | EER: {valid_eer:.4f}%")
    metrics.update({
        "valid_ret_acc": valid_ret_acc,
        "valid_eer": valid_eer,
    })

    # -----------------------------
    # Combined Score Calculation
    # -----------------------------
    valid_combined_score = 0.5 * valid_cls_acc + 0.5 * (100.0 - valid_eer)
    print(f"Val (Combined) | Score: {valid_combined_score:.4f}%")
    metrics.update({
        "valid_combined_score": valid_combined_score
    })

    # -----------------------------
    # Log metrics
    # -----------------------------
    if "run" in globals() and run is not None:
        run.log(metrics, step=epoch)

    # -----------------------------
    # Save checkpoints (all inside checkpoints/<run_name>/)
    # -----------------------------
    # Save best classification model
    if valid_cls_acc >= best_cls_acc:
        best_cls_acc = valid_cls_acc
        best_cls_path = os.path.join(config["checkpoint_dir"], "best_cls.pth")
        save_model(model, optimizer, scheduler, metrics, epoch, best_cls_path, ema)
        print(f"Saved best classification model: {best_cls_path}")

    # Save best retrieval model
    if valid_ret_acc >= best_ret_acc:
        best_ret_acc = valid_ret_acc
        best_ret_path = os.path.join(config["checkpoint_dir"], "best_ret.pth")
        save_model(model, optimizer, scheduler, metrics, epoch, best_ret_path, ema)
        print(f"Saved best retrieval model: {best_ret_path}")

    # Save best combined model
    if valid_combined_score >= best_combined_score:
        best_combined_score = valid_combined_score
        best_combined_path = os.path.join(config["checkpoint_dir"], "best_combined.pth")
        save_model(model, optimizer, scheduler, metrics, epoch, best_combined_path, ema)
        if "run" in globals() and run is not None:
            wandb.save(best_combined_path, base_path=os.path.dirname(config["checkpoint_dir"]))
        print(f"Saved best combined model: {best_combined_path}")

    # Last checkpoint carries the running bests so a resumed session continues correctly
    metrics.update({
        "best_cls_acc": best_cls_acc,
        "best_ret_acc": best_ret_acc,
        "best_combined_score": best_combined_score,
    })
    save_model(model, optimizer, scheduler, metrics, epoch, last_path, ema)
    print(f"Saved last epoch model: {last_path}")
    epoch_time = time.time() - epoch_t0
    print(f"Epoch time {epoch_time / 60:.1f} min | elapsed {(time.time() - NOTEBOOK_START) / 3600:.2f}h")

## Inference and submission file

In [ ]:
# Offline copy of https://github.com/CMU-IDeeL/HW2P2-Student-Utils (utils/), unmodified.
# Used only when git clone fails (Internet off).
COURSE_UTILS = {
    '__init__.py': 'from .metrics_calculator import MetricsCalculator\nfrom .submission_utils import (\n    get_public_indices,\n    build_usage_values,\n    load_cls_image_paths_file_order,\n    run_cls_inference_file_order,\n    run_ver_inference,\n    build_component_rows,\n    build_merged_submission_rows,\n    write_component_csv,\n    write_submission_csv,\n    write_submission_bundle,\n)\n',
    'metrics_calculator.py': 'import torch\nimport numpy as np\nfrom typing import Dict, List, Union\nfrom scipy.interpolate import interp1d\nfrom scipy.optimize import brentq\nimport sklearn.metrics as mt\n\nclass MetricsCalculator:\n    """\n    A unified class for calculating both classification and verification metrics.\n\n    Supports:\n    - Classification: Accuracy, Top-K Accuracy\n    - Verification: EER, AUC, ACC, TPR@FPR\n    - Running averages for all metrics\n    """\n\n    def __init__(self, FPRs: List[float] = [1e-4, 5e-4, 1e-3, 5e-3, 5e-2]):\n        """\n        Initialize metrics calculator.\n\n        Args:\n            FPRs: List of FPR values at which to calculate TPR\n        """\n        self.FPRs = FPRs\n        self.reset()\n\n    def reset(self) -> None:\n        """Reset all metric counters."""\n        self.cls_metrics = {\n            \'top1\': {\'sum\': 0, \'count\': 0},\n            \'top5\': {\'sum\': 0, \'count\': 0}\n        }\n        self.ver_metrics = {\n            \'ACC\': {\'sum\': 0, \'count\': 0},\n            \'EER\': {\'sum\': 0, \'count\': 0},\n            \'AUC\': {\'sum\': 0, \'count\': 0}\n        }\n        for fpr in self.FPRs:\n            self.ver_metrics[f\'TPR@FPR={fpr}\'] = {\'sum\': 0, \'count\': 0}\n\n    def update_classification(self,\n                            outputs: torch.Tensor,\n                            targets: torch.Tensor) -> Dict[str, float]:\n        """\n        Update classification metrics.\n\n        Args:\n            outputs: Model outputs (N, num_classes)\n            targets: Ground truth labels (N,)\n\n        Returns:\n            Dictionary containing current top1 and top5 accuracies\n        """\n        batch_size = targets.size(0)\n        topk = (1, 5)\n        maxk = min(max(topk), outputs.size(1))\n\n        # Get top-k predictions\n        _, pred = outputs.topk(maxk, 1, True, True)\n        pred = pred.t()\n        correct = pred.eq(targets.view(1, -1).expand_as(pred))\n\n        # Calculate accuracies\n        results = {}\n        for k, name in zip(topk, [\'top1\', \'top5\']):\n            correct_k = correct[:k].reshape(-1).float().sum(0)\n            accuracy = correct_k * 100.0 / batch_size\n            self.cls_metrics[name][\'sum\'] += accuracy.item() * batch_size\n            self.cls_metrics[name][\'count\'] += batch_size\n            results[name] = accuracy.item()\n\n        return results\n\n    def update_verification(\n        self,\n        scores: Union[List[float], np.ndarray],\n        labels: Union[List[int], np.ndarray],\n        update_threshold: bool = False,\n    ) -> Dict[str, float]:\n        """\n        Update verification metrics.\n\n        Args:\n            scores: Similarity scores between pairs\n            labels: Ground truth labels (1 for match, 0 for non-match)\n\n        Returns:\n            Dictionary containing:\n            - ACC: Overall accuracy\n            - EER: Equal Error Rate\n            - AUC: Area Under Curve\n            - TPR@FPR=x: True Positive Rate at specific False Positive Rates\n        """\n        # Convert inputs to numpy arrays if needed\n        scores = np.array(scores)\n        labels = np.array(labels)\n\n        # Calculate ROC curve\n        fpr, tpr, thresholds = mt.roc_curve(labels, scores, pos_label=1)\n        roc_curve = interp1d(fpr, tpr)\n\n        # Calculate metrics\n        results = {}\n\n        # EER\n        eer = 100. * brentq(lambda x: 1. - x - roc_curve(x), 0., 1.)\n        self.ver_metrics[\'EER\'][\'sum\'] += eer * len(labels)\n        self.ver_metrics[\'EER\'][\'count\'] += len(labels)\n        results[\'EER\'] = eer\n\n        # Find threshold that gives EER\n        eer_threshold_idx = np.nanargmin(np.abs(fpr - (1 - tpr)))\n        optimal_threshold = thresholds[eer_threshold_idx]\n        results[\'threshold\'] = optimal_threshold\n\n        # AUC\n        auc = 100. * mt.auc(fpr, tpr)\n        self.ver_metrics[\'AUC\'][\'sum\'] += auc * len(labels)\n        self.ver_metrics[\'AUC\'][\'count\'] += len(labels)\n        results[\'AUC\'] = auc\n\n        # Accuracy\n        tnr = 1. - fpr\n        pos_num = np.sum(labels == 1)\n        neg_num = np.sum(labels == 0)\n        acc = 100. * max(tpr * pos_num + tnr * neg_num) / len(labels)\n        self.ver_metrics[\'ACC\'][\'sum\'] += acc * len(labels)\n        self.ver_metrics[\'ACC\'][\'count\'] += len(labels)\n        results[\'ACC\'] = acc\n\n        # TPR @ FPR\n        for fpr_target in self.FPRs:\n            tpr_at_fpr = 100. * roc_curve(float(fpr_target))\n            metric_name = f\'TPR@FPR={fpr_target}\'\n            self.ver_metrics[metric_name][\'sum\'] += tpr_at_fpr * len(labels)\n            self.ver_metrics[metric_name][\'count\'] += len(labels)\n            results[metric_name] = tpr_at_fpr\n\n        return results\n\n    def get_averages(self) -> Dict[str, Dict[str, float]]:\n        """Get running averages for all metrics."""\n        averages = {\n            \'classification\': {},\n            \'verification\': {}\n        }\n\n        # Classification metrics\n        for name, metric in self.cls_metrics.items():\n            if metric[\'count\'] > 0:\n                averages[\'classification\'][name] = metric[\'sum\'] / metric[\'count\']\n\n        # Verification metrics\n        for name, metric in self.ver_metrics.items():\n            if metric[\'count\'] > 0:\n                averages[\'verification\'][name] = metric[\'sum\'] / metric[\'count\']\n\n        return averages',
    'submission_utils.py': '"""Utilities for generating HW2P2 Kaggle submissions."""\n\nimport csv\nimport os\nimport random\nfrom typing import List, Sequence\n\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom torchvision.io import decode_image\nfrom tqdm import tqdm\n\nPUBLIC_RATIO = 0.40\nRANDOM_SEED = 11785\n\nTASK_CLS = "classification"\nTASK_VER = "verification"\n\nSUBMISSION_COLUMNS = ["ID", "Task", "Prediction", "Usage"]\nCOMPONENT_COLUMNS = ["ID", "Label", "Usage"]\n\n\ndef get_public_indices(num_samples: int) -> set:\n    """Return indices assigned to the public leaderboard split."""\n    random.seed(RANDOM_SEED)\n    num_public = int(num_samples * PUBLIC_RATIO)\n    return set(random.sample(range(num_samples), num_public))\n\n\ndef build_usage_values(num_samples: int) -> List[str]:\n    public_indices = get_public_indices(num_samples)\n    return ["Public" if i in public_indices else "Private" for i in range(num_samples)]\n\n\n# def load_cls_image_paths_file_order(cls_test_root: str) -> List[str]:\n#     """\n#     Load cls test image paths in labels.txt file order.\n\n#     Submission IDs follow labels.txt line order (test_000001.jpg -> id 0),\n#     not the label-sorted order used by ImageDataset.\n#     """\n#     labels_file = os.path.join(cls_test_root, "labels.txt")\n#     images_dir = os.path.join(cls_test_root, "images")\n#     image_paths = []\n\n#     with open(labels_file, "r") as f:\n#         for line in f:\n#             img_name = line.strip().split()[0]\n#             image_paths.append(os.path.join(images_dir, img_name))\n\n#     return image_paths\n\ndef load_cls_image_paths_file_order(cls_test_root: str) -> List[str]:\n    """\n    Load classification test image paths in numeric filename order.\n\n    Missing indices are allowed because duplicate images were removed\n    during dataset deduplication.\n\n    The deduplicated classification test set must contain exactly\n    43,097 images.\n    """\n    images_dir = os.path.join(cls_test_root, "images")\n\n    if not os.path.isdir(images_dir):\n        raise FileNotFoundError(f"Image directory not found: {images_dir}")\n\n    image_names = [\n        name\n        for name in os.listdir(images_dir)\n        if name.lower().endswith((".jpg", ".jpeg", ".png"))\n    ]\n\n    if not image_names:\n        raise RuntimeError(f"No images found in: {images_dir}")\n\n    # Extract the original numeric index from test_XXXXXX.jpg\n    try:\n        numbers_and_names = [\n            (\n                int(os.path.splitext(name)[0].split("_")[-1]),\n                name,\n            )\n            for name in image_names\n        ]\n    except ValueError as e:\n        raise ValueError(\n            "Classification test filenames must follow the format "\n            "test_XXXXXX.jpg."\n        ) from e\n\n    # Keep inference order consistent with the original numeric file order.\n    numbers_and_names.sort(key=lambda x: x[0])\n\n    numbers = [number for number, _ in numbers_and_names]\n    image_names = [name for _, name in numbers_and_names]\n\n    # Exact expected size of the deduplicated test set.\n    EXPECTED_NUM_IMAGES = 43097\n\n    assert len(image_names) == EXPECTED_NUM_IMAGES, (\n        f"Expected {EXPECTED_NUM_IMAGES:,} classification test images "\n        f"after deduplication, but found {len(image_names):,}."\n    )\n\n    # Missing indices are allowed, but duplicate numeric IDs are not.\n    assert len(numbers) == len(set(numbers)), (\n        "Duplicate classification test image indices found."\n    )\n\n    print(f"Loaded {len(image_names):,} classification test images")\n    print("First 3:", image_names[:3])\n    print("Last 3: ", image_names[-3:])\n\n    return [\n        os.path.join(images_dir, name)\n        for name in image_names\n    ]\n\n@torch.no_grad()\ndef run_cls_inference_file_order(\n    model,\n    cls_test_root: str,\n    transform,\n    device,\n    batch_size: int = 64,\n    logits_key: str = "cls_output",\n) -> List[int]:\n    # """Run classification inference in labels.txt file order."""\n    """Run classification inference in sorted filename order."""\n    model.eval()\n    image_paths = load_cls_image_paths_file_order(cls_test_root)\n    predictions = []\n\n    for start in tqdm(range(0, len(image_paths), batch_size), desc="Test (Cls)", leave=False):\n        batch_paths = image_paths[start : start + batch_size]\n        images = []\n        for path in batch_paths:\n            image = decode_image(path, mode="RGB") / 255.0\n            images.append(transform(image))\n\n        images = torch.stack(images).to(device, non_blocking=True)\n        outputs = model(images)\n        logits = outputs[logits_key]\n        batch_preds = logits.argmax(dim=1).cpu().tolist()\n        predictions.extend(batch_preds)\n\n    return predictions\n\n\n@torch.no_grad()\ndef run_ver_inference(\n    model,\n    pair_dataloader: DataLoader,\n    device,\n    embedding_key: str = "embedding",\n) -> List[float]:\n    """Run verification inference and return cosine similarity scores."""\n    model.eval()\n    scores = []\n\n    progress = tqdm(pair_dataloader, desc="Test (Veri)", dynamic_ncols=True, leave=False)\n\n    for batch in progress:\n        if len(batch) == 3:\n            images1, images2, _ = batch\n        else:\n            images1, images2 = batch\n\n        images1 = images1.to(device, non_blocking=True)\n        images2 = images2.to(device, non_blocking=True)\n\n        images = torch.cat([images1, images2], dim=0)\n        outputs = model(images)\n        feats = F.normalize(outputs[embedding_key], dim=1)\n\n        feats1, feats2 = feats.chunk(2, dim=0)\n        similarity = F.cosine_similarity(feats1, feats2, dim=1)\n        scores.extend(similarity.cpu().numpy().tolist())\n\n    return scores\n\n\ndef build_component_rows(predictions: Sequence, usage_values: Sequence[str]) -> List[dict]:\n    return [\n        {"ID": i, "Label": pred, "Usage": usage}\n        for i, (pred, usage) in enumerate(zip(predictions, usage_values))\n    ]\n\n\ndef build_merged_submission_rows(\n    cls_predictions: Sequence[int],\n    ver_scores: Sequence[float],\n) -> List[dict]:\n    """Build merged submission rows matching merged_solution.csv structure."""\n    cls_usage = build_usage_values(len(cls_predictions))\n    ver_usage = build_usage_values(len(ver_scores))\n    cls_id_offset = len(cls_predictions)\n    rows = []\n\n    for i, pred in enumerate(cls_predictions):\n        rows.append(\n            {\n                "ID": i,\n                "Task": TASK_CLS,\n                "Prediction": pred,\n                "Usage": cls_usage[i],\n            }\n        )\n\n    for i, score in enumerate(ver_scores):\n        rows.append(\n            {\n                "ID": cls_id_offset + i,\n                "Task": TASK_VER,\n                "Prediction": score,\n                "Usage": ver_usage[i],\n            }\n        )\n\n    return rows\n\n\ndef write_component_csv(path: str, rows: List[dict]) -> None:\n    with open(path, "w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=COMPONENT_COLUMNS)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef write_submission_csv(path: str, rows: List[dict]) -> None:\n    with open(path, "w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=SUBMISSION_COLUMNS)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef write_submission_bundle(\n    cls_predictions: Sequence[int],\n    ver_scores: Sequence[float],\n    submission_path: str = "submission.csv",\n    cls_component_path: str = "cls_submission.csv",\n    ver_component_path: str = "ver_submission.csv",\n) -> List[dict]:\n    """Write component CSVs and merged submission CSV."""\n    cls_usage = build_usage_values(len(cls_predictions))\n    ver_usage = build_usage_values(len(ver_scores))\n\n    cls_rows = build_component_rows(cls_predictions, cls_usage)\n    ver_rows = build_component_rows(ver_scores, ver_usage)\n    merged_rows = build_merged_submission_rows(cls_predictions, ver_scores)\n\n    write_component_csv(cls_component_path, cls_rows)\n    write_component_csv(ver_component_path, ver_rows)\n    write_submission_csv(submission_path, merged_rows)\n\n    return merged_rows\n',
}

In [ ]:
#### DO NOT MODIFY ####
import os, shutil, subprocess
if not os.path.isdir("utils"):  # fetch once; cross-platform replacement for !mv / !rm
    try:
        subprocess.run(["git", "clone", "-q", "https://github.com/CMU-IDeeL/HW2P2-Student-Utils.git"], check=True, timeout=120)
        shutil.move("HW2P2-Student-Utils/utils", "utils")
        shutil.rmtree("HW2P2-Student-Utils", ignore_errors=True)
    except Exception as e:  # Internet off: use the copy embedded in this notebook
        print("git clone failed, using the embedded copy of the course utils:", e)
        os.makedirs("utils", exist_ok=True)
        for _fname, _src in COURSE_UTILS.items():
            with open(os.path.join("utils", _fname), "w") as f:
                f.write(_src)
from utils import (
    run_cls_inference_file_order,
    run_ver_inference,
    write_submission_bundle,
)

# Training is finished: free the optimizer state, EMA copy and cached blocks before inference
for _name in ("optimizer", "scheduler", "scaler", "ema", "train_model"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

# Load this run's best combined checkpoint and evaluate with flip TTA
model, _, _, best_epoch, _ = load_model(
    model, path=os.path.join(config["checkpoint_dir"], "best_combined.pth"), device=DEVICE
)
best_ckpt = torch.load(os.path.join(config["checkpoint_dir"], "best_combined.pth"), map_location="cpu")
if best_ckpt.get("ema_state_dict") is not None:
    model.load_state_dict(best_ckpt["ema_state_dict"])  # the EMA weights are the ones that were validated
    print("Using the EMA weights")
del best_ckpt
tta_model = FlipTTA(model).eval()
tta_ver = valid_epoch_ver(tta_model, ver_val_loader, DEVICE)
print(f"Best epoch {best_epoch + 1} | Val EER with flip TTA: {tta_ver['EER']:.4f}%")

cls_predictions = run_cls_inference_file_order(
    model=tta_model,
    cls_test_root=cls_data_dir + "/test",
    transform=val_transforms,
    device=DEVICE,
    batch_size=config["batch_size"],
    logits_key="out",
)

ver_scores = run_ver_inference(
    model=tta_model,
    pair_dataloader=ver_test_loader,
    device=DEVICE,
    embedding_key="feats",
)

submission_rows = write_submission_bundle(
    cls_predictions=cls_predictions,
    ver_scores=ver_scores,
    submission_path=os.path.join(config["checkpoint_dir"], "submission.csv"),
    cls_component_path=os.path.join(config["checkpoint_dir"], "cls_submission.csv"),
    ver_component_path=os.path.join(config["checkpoint_dir"], "ver_submission.csv"),
)

shutil.copy(os.path.join(config["checkpoint_dir"], "submission.csv"), "submission.csv")
print(f"Saved {config['checkpoint_dir']}/submission.csv (copied to ./submission.csv)")
print(f"  Classification predictions: {len(cls_predictions)}")
print(f"  Verification scores:        {len(ver_scores)}")
print(f"  Total rows:                  {len(submission_rows)}")
print(pd.DataFrame(submission_rows).head())

In [ ]:
if SMOKE:
    os.remove("submission.csv")  # the smoke copy must never be mistaken for a real submission
    print("SMOKE TEST PASSED")
elif SUBMIT_TO_KAGGLE:
    kaggle_api().competition_submit("submission.csv", f"{config['run_name']} (Kaggle notebook)", COMPETITION)
    print("Submitted to Kaggle")
print("Done. Outputs in", os.path.abspath(config["checkpoint_dir"]))